# Modèle de prix

En plus de l'analyse des retards, il est demandé de créer une API qui propose un prix journalier de location à partir des caractéristiques d'une voiture. Cette seconde partie repose sur un autre jeu de données, get_around_pricing_project.csv, sans lien avec le premier. Elle est donc traitée séparément dans ce notebook.

Dans ce notebook, nous cherchons à comprendre les données et à choisir le modèle. L'entraînement final se fera dans un script, src/train.py, avec un suivi MLflow, et le modèle obtenu sera embarqué dans l'API.

## Problématique
Lorsqu'un propriétaire met sa voiture en location sur Getaround, il doit fixer un prix journalier, mais n'a aucun repère pour prendre cette décision. Le jeu de données rassemble 4 843 véhicules déjà en location avec leurs caractéristiques et leur prix. L'objectif est d'apprendre, à partir de ces exemples, la relation entre les caractéristiques d'une voiture et son prix, afin de suggérer un prix pour un véhicule qui n'est pas dans la base. Il s'agit d'une régression.
La cible du modèle est le prix journalier ; les entrées sont les 13 caractéristiques de la voiture. Le modèle sera exposé par une API, appelable depuis l'application au moment de créer l'annonce.

## Exploration des données

In [1]:
import pandas as pd
import plotly.express as px

In [2]:
pricing = pd.read_csv("data/raw/get_around_pricing_project.csv", index_col=0)
print(pricing.shape)
pricing.head(10)

(4843, 14)


,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183
5,Citroën,152352,225,petrol,black,convertible,True,True,False,False,True,True,True,131
6,Citroën,205219,145,diesel,grey,convertible,True,True,False,False,True,True,True,111
7,Citroën,115560,105,petrol,white,convertible,True,True,False,False,False,True,True,78
8,Peugeot,123886,125,petrol,black,convertible,True,False,False,False,False,True,True,79
9,Citroën,139541,135,diesel,white,convertible,False,False,False,False,True,False,True,132


In [3]:
print(pricing.isna().sum().sum())
pricing.describe()

0


,mileage,engine_power,rental_price_per_day
count,4.843000e+03,4843.00000,4843.000000
mean,1.409628e+05,128.98823,121.214536
std,6.019674e+04,38.99336,33.568268
min,-6.400000e+01,0.00000,10.000000
25%,1.029135e+05,100.00000,104.000000
50%,1.410800e+05,120.00000,119.000000
75%,1.751955e+05,135.00000,136.000000
max,1.000376e+06,423.00000,422.000000


Le fichier ne contient aucune valeur manquante.   

Il recense un total de 4843 véhicules.   
Il n'est toutefois pas exempt de nettoyage: on observe notamment un kilométrage minimal négatif, une puissance de moteur égale à 0 ou encore un véhicule à plus d'un million de kilomètres . Ces valeurs ne sont pas réalistes et vont nécessiter un travail d'identification de valeurs potentiellement erronnées pour préparer le dataframe. 

In [4]:
pricing["mileage"].nlargest(10)

3732    1000376
557      484615
2350     477571
2829     439060
3198     405816
1573     400654
1003     398043
2912     394135
1521     391309
3554     388616
Name: mileage, dtype: int64

On constate qu'il y a seulement un véhicule concerné par un kilométrage très élevé, à plus de 1 million de kilomètres, alors que le deuxième véhicule avec le kilométrage le plus élevé est à 484 000 km. En conséquence, nous pouvons présumer une erreur de frappe, peut-être un zéro en trop, et nous allons supprimer cette entrée. Un seuil à 500 000 km permet de conserver toutes les entrées existantes à l'exception de celle-ci. 

In [5]:
pricing[(pricing["mileage"] < 0) | (pricing["mileage"] > 500_000) | (pricing["engine_power"] == 0)]

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
2938,Renault,-64,230,diesel,black,sedan,True,True,False,True,False,False,True,274
3732,Citroën,1000376,90,diesel,black,subcompact,True,False,False,False,False,False,True,37
3765,Nissan,81770,0,diesel,white,suv,False,False,False,False,False,False,False,108


Un total de 3 lignes sur 4843 sont à supprimer pour corriger des valeurs aberrantes. 

In [6]:
pricing = pricing[(pricing["mileage"] >= 0) & (pricing["mileage"] <= 500_000) & (pricing["engine_power"] > 0)]
pricing.shape

(4840, 14)

### Variété des marques

In [7]:
pricing["model_key"].value_counts()

model_key
Citroën        968
Renault        915
BMW            827
Peugeot        642
Audi           526
Nissan         274
Mitsubishi     231
Mercedes        97
Volkswagen      65
Toyota          53
SEAT            46
Subaru          44
Opel            33
Ferrari         33
PGO             33
Maserati        18
Suzuki           8
Porsche          6
Ford             5
KIA Motors       3
Alfa Romeo       3
Fiat             2
Lexus            2
Lamborghini      2
Mini             1
Mazda            1
Honda            1
Yamaha           1
Name: count, dtype: int64

### Distribution des prix

Nous allons nous intéresser à la distribution des prix en fonction des véhicules. 

In [8]:
fig = px.histogram(
    pricing,
    x="rental_price_per_day",
    nbins=60,
    title="Distribution du prix journalier",
    labels={"rental_price_per_day": "Prix par jour (€)"},
)
fig.show()

Nous observons une courbe en cloche centrée autour de 110 à 120 € par jour. 
La courbe est étirée sur la droite, avec quelques voitures dont les prix vont au-delà de 200, voire même 300 € par jour. À l'inverse, très peu de voitures sont sous la barre des 50 € par jour. 

## Modélisation

### Préparation des données 

Nous allons d'abord séparer les caractéristiques de la cible et ensuite répartir le jeu de données en deux parties : 80/20. 80 % seront utilisés pour l'entraînement et 20 % seront mis de côté pour évaluer le modèle. Le tirage est fixé de façon à pouvoir être reproduit. C'est le rôle de random_state. 

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = pricing.drop(columns="rental_price_per_day")
y = pricing["rental_price_per_day"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(3872, 13) (968, 13)


Les colonnes ne sont pas toutes de même type et doivent donc être traitées séparément. Elles peuvent se répartir en trois groupes :
- colonne de texte
- colonne numérique
- colonne d'options vrai-faux

In [10]:
categorical = ["model_key", "fuel", "paint_color", "car_type"]
numeric = ["mileage", "engine_power"]
boolean = [col for col in X.columns if col not in categorical + numeric]
print(boolean)

['private_parking_available', 'has_gps', 'has_air_conditioning', 'automatic_car', 'has_getaround_connect', 'has_speed_regulator', 'winter_tires']


Au niveau du préprocesseur, on applique un encodage one-hot aux colonnes de texte, une standardisation aux colonnes numériques, et on laisse les options telles quelles, d'où l'usage d'un passthrough. Une valeur inconnue à l'entraînement, par exemple une marque, est encodée par des zéros plutôt que de provoquer une erreur (handle_unknown="ignore"). Le préprocesseur sera intégré à un pipeline avec chaque modèle ; ainsi l'API n'a aucune transformation à effectuer avant de prédire.

Ce paramétrage suppose de ne pas utiliser drop="first" : le vecteur nul désignerait alors la première catégorie, et une valeur inconnue serait confondue avec elle. La colinéarité qui en résulte est sans effet sur une forêt aléatoire ; pour la régression linéaire, elle n'affecte que l'interprétation des coefficients, pas les prédictions.

In [11]:
preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),  # pas de drop=first, cas des modeles inconus.
    ("numeric", StandardScaler(), numeric),
    ("boolean", "passthrough", boolean),
])

## Comparaison de deux modèles

Nous allons étudier un modèle de régression linéaire sur les caractéristiques brutes. Toutefois, la linéarité de la relation entre les caractéristiques et le prix n'étant pas établie, il convient d'analyser également un modèle de type Random Forest, plus susceptible de capter d'éventuelles non-linéarités par rapport aux variables, courbures ou interactions entre caractéristiques, sans que nous ayons à les spécifier.

Nous allons donc étudier deux modèles et les comparer :

- une régression linéaire ;
- une forêt aléatoire (Random Forest).  
  
Les deux modèles suivent le même cheminement, dans une même boucle : même préprocesseur, même découpage, mêmes mesures. Pour chacun, nous relevons le R² sur l'entraînement et sur le test, et l'erreur absolue moyenne sur le test, en euros.

La comparaison entre les deux modèles se fera sur deux mesures : le R² et le MAE. 

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(random_state=42),
}

results = []
for name, model in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor), ("regressor", model)])
    pipeline.fit(X_train, y_train)
    results.append({
        "model": name,
        "r2_train": r2_score(y_train, pipeline.predict(X_train)),
        "r2_test": r2_score(y_test, pipeline.predict(X_test)),
        "mae_test": mean_absolute_error(y_test, pipeline.predict(X_test)),
    })

pd.DataFrame(results).round(3)

,model,r2_train,r2_test,mae_test
0,linear_regression,0.714,0.705,12.398
1,random_forest,0.964,0.746,10.989


Sur le jeu de test, la forêt aléatoire prédit mieux que la régression linéaire sur les deux mesures : R² et MAE. Elle explique en effet 74,6 % de la variation des prix, contre 70,5 % pour la régression linéaire, avec une erreur absolue moyenne de 11 € contre 12,4 €, Pour un prix médian de 119 € . 

Une explication possible à cet écart est que la relation réelle entre caractéristiques et prix  inclut peut-être des courbures ou des interactions entre caractéristiques que la régression linéaire brute ne capte pas. L'écart reste toutefois modeste et l'essentiel du prix s'explique par Une simple combinaison linéaire des variables. 

La forêt aléatoire montre un surapprentissage significatif avec un R2 de 0.964 à l'entrainement alors qu'il n'est que de 0.746 au test. Cet écart est attendu avec les réglages par défaut, où les arbres grandissent jusqu'à isoler presque chaque voiture d'entraînement. La régression linéaire, quant à elle, n'a que faiblement overfitté: 0.714 contre 0.705.

Dans la cellule ci-dessous, nous allons non seulement ajouter un modele (random forest avec profondeur de 10) mais nous allons aussi faire une validation croisee sur l'ensemble des 3 modeles: linear regression, random forest, et random forest depth 10. Par défaut, les arbres ne sont pas limités en profondeur et finissent par isoler chaque voiture. C'est essentiellement cela qui explique le surapprentissage. En limitant la profondeur, nous devrions pouvoir le reduire.
De plus, en evaluant les modeles par validation croisee, la comparaison ne dependra plus d'un seul decoupage. La validation croisee se fera sur 5 part (k=5).

In [13]:
from sklearn.model_selection import KFold, cross_validate

models["random_forest_depth_10"] = RandomForestRegressor(max_depth=10, random_state=42)
folds = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []
for name, model in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor), ("regressor", model)])
    scores = cross_validate(pipeline, X, y, cv=folds, scoring=["r2", "neg_mean_absolute_error"], return_train_score=True)
    cv_results.append({
        "model": name,
        "r2_train": scores["train_r2"].mean(),
        "r2_test": scores["test_r2"].mean(),
        "mae_test": -scores["test_neg_mean_absolute_error"].mean(),
    })

pd.DataFrame(cv_results).round(3)

,model,r2_train,r2_test,mae_test
0,linear_regression,0.717,0.703,12.307
1,random_forest,0.965,0.755,10.576
2,random_forest_depth_10,0.887,0.748,10.875


La validation croisée confirme le classement obtenu sur un seul découpage : la forêt aléatoire ameliore la mae d'environ 1,5 € d'erreur moyenne par rapport à la régression linéaire.

Le modèle dont la profondeur des arbres est limitée à 10 donne des résultats sensiblement identiques à la forêt par défaut sur le test : un R² de 0,748 contre 0,755, et seulement 0,3 € de plus pour la MAE. En échange, le surapprentissage est nettement réduit : le R² d'entraînement passe de 0,965 à 0,887.

Nous retenons ce modèle.